# ML - BOOSTING

In [69]:
import numpy as np
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from utils1 import get_classifier_metrics
from sklearn.model_selection import GridSearchCV
from collections import Counter
from sklearn.metrics import classification_report

## Paso 1. Lectura del conjunto de datos procesado

In [3]:
# Cargamos los dataframes 
with open('../data/processed/04_df_invalids_removed.pkl', 'rb') as f:
    df_invalids_removed = pickle.load(f)

with open('../data/processed/04_df_invalids_mode.pkl', 'rb') as f:
    df_invalids_mode = pickle.load(f)

with open('../data/processed/04_df_invalids_knn.pkl', 'rb') as f:
    df_invalids_knn = pickle.load(f)

## Paso 2. Split

In [4]:
X_invalids_removed = df_invalids_removed.drop('Outcome', axis= 1)
y_invalids_removed = df_invalids_removed['Outcome']

X_invalids_mode= df_invalids_mode.drop('Outcome', axis= 1)
y_invalids_mode = df_invalids_mode['Outcome']

X_invalids_knn= df_invalids_knn.drop('Outcome', axis= 1)
y_invalids_knn = df_invalids_knn['Outcome']

X_train_1, X_test_1, y_train_1, y_test_1 = train_test_split(X_invalids_removed, y_invalids_removed, test_size=0.2, random_state=21)
X_train_2, X_test_2, y_train_2, y_test_2 = train_test_split(X_invalids_mode, y_invalids_mode, test_size=0.2, random_state=21)
X_train_3, X_test_3, y_train_3, y_test_3 = train_test_split(X_invalids_knn, y_invalids_knn, test_size=0.2, random_state=21)

## Paso 3. Modelado y Ajuste

In [5]:
model_boosting_default_1 = XGBClassifier(random_state=21)
model_boosting_default_1.fit(X_train_1, y_train_1)

model_boosting_default_2 = XGBClassifier(random_state=21)
model_boosting_default_2.fit(X_train_2, y_train_2)

model_boosting_default_3 = XGBClassifier(random_state=21)
model_boosting_default_3.fit(X_train_3, y_train_3)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


## Paso 4. Predicción

In [6]:
y_pred_test_1 = model_boosting_default_1.predict(X_test_1)
y_pred_train_1 = model_boosting_default_1.predict(X_train_1)

y_pred_test_2 = model_boosting_default_2.predict(X_test_2)
y_pred_train_2 = model_boosting_default_2.predict(X_train_2)

y_pred_test_3 = model_boosting_default_3.predict(X_test_3)
y_pred_train_3 = model_boosting_default_3.predict(X_train_3)

In [7]:
default_model_metrics_1 = get_classifier_metrics(y_pred_test_1, y_test_1, y_pred_train_1, y_train_1, average='weighted')
default_model_metrics_1

,Accuracy,F1 Score,Precision,Recall
Train set,1.000000,1.000000,1.000000,1.000000
Test set,0.734177,0.723725,0.726029,0.734177


In [8]:
default_model_metrics_2 = get_classifier_metrics(y_pred_test_2, y_test_2, y_pred_train_2, y_train_2, average='weighted')
default_model_metrics_2

,Accuracy,F1 Score,Precision,Recall
Train set,1.000000,1.000000,1.000000,1.000000
Test set,0.701299,0.680869,0.699905,0.701299


In [9]:
default_model_metrics_3 = get_classifier_metrics(y_pred_test_3, y_test_3, y_pred_train_3, y_train_3, average='weighted')
default_model_metrics_3

,Accuracy,F1 Score,Precision,Recall
Train set,1.000000,1.000000,1.000000,1.000000
Test set,0.701299,0.685789,0.696685,0.701299


### Modelo optimizado con los hiperparámetros con `GridSearchCV`

In [ ]:
'''n_estimators'# número de árboles
    'max_depth' # profundidad máxima de cada árbol
    'learning_rate': # tasa de aprendizaje
    'subsample': # proporción de muestras usadas para entrenar cada árbol
    'colsample_bytree':  # proporción de características usadas por árbol
    'gamma': # regularización: requiere mejora mínima de pérdida para dividir
    'reg_alpha': # L1 regularization (sparse features)
    'reg_lambda': # L2 regularization (default is 1)

IndentationError: unexpected indent (1471930609.py, line 2)

In [73]:
# Calcular manualmente el peso para balancear clases (0: no diabetes, 1: diabetes)

counts_1 = Counter(y_train_1)
neg_1, pos_1 = counts_1[0], counts_1[1]

counts_3 = Counter(y_train_3)
neg_3, pos_3 = counts_3[0], counts_3[1]

scale_pos_weight_1 = neg_1 / pos_1
scale_pos_weight_3 = neg_3 / pos_3


In [78]:
param_grid_1 = {'n_estimators': [10, 20, 30], 
              'max_depth' : [3, 5, 7],
              'learning_rate': [0.01, 0.05, 0.1],          
              'subsample': [0.8, 1.0],                    
              'colsample_bytree': [0.8, 1.0],              
              'gamma': [0, 0.5, 1],
              'scale_pos_weight': [1, scale_pos_weight_1]}
  

param_grid_3 = {'n_estimators': [20, 30], 
              'max_depth' : [3, 5, 7],
              'learning_rate': [0.01, 0.05, 0.1],          
              'subsample': [0.8, 1.0],                    
              'colsample_bytree': [0.8, 1.0],              
              'gamma': [0, 0.5, 1],
              'scale_pos_weight': [1, scale_pos_weight_3]}
        

grid_1 = GridSearchCV(model_boosting_default_1,
                      param_grid_1,
                      scoring='recall',
                      cv=5)

grid_3 = GridSearchCV(model_boosting_default_3,
                      param_grid_3,
                      scoring='recall',
                      cv=5)                                    

In [76]:
# Entrenamos el grid con los hiperparametros
grid_1.fit(X_train_1, y_train_1)
#Devuelvemos los mejores parametros despues de entrenarlo
grid_1.best_params_

{'colsample_bytree': 1.0,
 'gamma': 0,
 'learning_rate': 0.01,
 'max_depth': 3,
 'n_estimators': 10,
 'reg_alpha': 0,
 'reg_lambda': 1.0,
 'scale_pos_weight': 2.0784313725490198,
 'subsample': 1.0}

In [77]:
# Entrenamos el grid con los hiperparametros
grid_3.fit(X_train_3, y_train_3)
#Devuelvemos los mejores parametros despues de entrenarlo
grid_3.best_params_

KeyboardInterrupt: 

In [63]:
#Modelos con los mejores parametros
grid_boosting_1 = grid_1.best_estimator_
grid_boosting_3 = grid_3.best_estimator_

In [64]:
# Repetimos el entrenamiento pero ahora con el grid que tiene los hiperparametros establecidos
grid_boosting_1.fit(X_train_1, y_train_1)
grid_boosting_3.fit(X_train_3, y_train_3)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,1.0
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [65]:
y_pred_test_1_grid = grid_boosting_1.predict(X_test_1)
y_pred_train_1_grid = grid_boosting_1.predict(X_train_1)

y_pred_test_3_grid = grid_boosting_3.predict(X_test_3)
y_pred_train_3_grid = grid_boosting_3.predict(X_train_3)

In [66]:
grid_boosting_model_metrics_1 = get_classifier_metrics(y_pred_test_1_grid, y_test_1, y_pred_train_1_grid, y_train_1, average='weighted')
grid_boosting_model_metrics_1 


,Accuracy,F1 Score,Precision,Recall
Train set,0.83121,0.834689,0.845726,0.83121
Test set,0.78481,0.783898,0.783250,0.78481


In [71]:
report_bm_1 = classification_report(y_test_1, y_pred_test_1_grid)
print(report_bm_1)

              precision    recall  f1-score   support

           0       0.83      0.84      0.83        51
           1       0.70      0.68      0.69        28

    accuracy                           0.78        79
   macro avg       0.77      0.76      0.76        79
weighted avg       0.78      0.78      0.78        79



In [67]:
grid_boosting_model_metrics_3 = get_classifier_metrics(y_pred_test_3_grid, y_test_3, y_pred_train_3_grid, y_train_3, average='weighted')
grid_boosting_model_metrics_3

,Accuracy,F1 Score,Precision,Recall
Train set,0.806189,0.810916,0.834722,0.806189
Test set,0.759740,0.758547,0.757947,0.759740


In [72]:
report_bm_3 = classification_report(y_test_3, y_pred_test_3_grid)
print(report_bm_3)

              precision    recall  f1-score   support

         0.0       0.79      0.82      0.81        94
         1.0       0.70      0.67      0.68        60

    accuracy                           0.76       154
   macro avg       0.75      0.74      0.75       154
weighted avg       0.76      0.76      0.76       154

